# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# I will build a feature vector from page-level search-performance and content attributes that are available before the prediction moment. Numeric features will be converted to numeric values and filled with median values. Categorical features will be converted using one-hot encoding. Outcome-derived and future-looking fields will not be included.



In [12]:
import os
import pandas as pd
import numpy as np

repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
]

categorical_features = [
    "content_type",
    "main_intent",
    "position_tier",
    "impression_tier",
]

numeric_features = [
    col for col in numeric_features
    if col in df.columns
]

categorical_features = [
    col for col in categorical_features
    if col in df.columns
]

X_numeric = df[numeric_features].apply(
    pd.to_numeric,
    errors="coerce"
).copy()

X_numeric = X_numeric.fillna(X_numeric.median())

X_categorical = pd.get_dummies(
    df[categorical_features].fillna("MISSING"),
    columns=categorical_features,
    dtype=int
)

X = pd.concat(
    [X_numeric, X_categorical],
    axis=1
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Feature matrix shape:", X.shape)
print("Remaining missing values:", int(X.isna().sum().sum()))
display(X.head())


Numeric features: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'word_count', 'content_age_days', 'days_since_last_update', 'engagement_rate', 'scroll_rate']
Categorical features: ['content_type', 'main_intent', 'position_tier', 'impression_tier']
Feature matrix shape: (30000, 27)
Remaining missing values: 0


,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,word_count,content_age_days,days_since_last_update,engagement_rate,scroll_rate,...,main_intent_transactional,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate
0,3803,29,17,0.76,10.6,3221.0,187,20,5.88,4.55,...,1,0,0,0,1,0,0,1,0,0
1,15320,7,9,0.05,20.3,2481.0,445,25,0.00,10.00,...,0,0,0,1,0,0,0,1,0,0
2,12581,11,11,0.09,36.5,3515.0,141,20,0.00,28.57,...,0,0,0,1,0,0,0,1,0,0
3,11751,58,78,0.49,6.2,2877.0,463,22,1.28,3.45,...,0,0,1,0,0,0,0,1,0,0
4,19140,24,145,0.13,44.0,2803.0,263,14,0.00,24.29,...,0,0,0,1,0,0,0,1,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Numeric features describe page performance and content characteristics. Missing numeric values are filled with the median of the same column. Categorical features are filled with the value MISSING and converted into one-hot columns. These features are assumed to be available before the prediction moment. The target outcome and future-looking fields are not used as features.


In [14]:
feature_notes = pd.DataFrame({
    "feature_group": ["numeric"] * len(numeric_features) +
                     ["categorical"] * len(categorical_features),
    "feature": numeric_features + categorical_features,
    "missing_handling": (
        ["median fill"] * len(numeric_features) +
        ["MISSING category"] * len(categorical_features)
    ),
    "available_before_prediction": [True] * (
        len(numeric_features) + len(categorical_features)
    )
})

display(feature_notes)

print("All selected features available before prediction:",
      feature_notes["available_before_prediction"].all())


,feature_group,feature,missing_handling,available_before_prediction
0,numeric,impressions_90d,median fill,True
1,numeric,clicks_90d,median fill,True
2,numeric,sessions_90d,median fill,True
3,numeric,ctr,median fill,True
4,numeric,avg_position,median fill,True
5,numeric,word_count,median fill,True
6,numeric,content_age_days,median fill,True
7,numeric,days_since_last_update,median fill,True
8,numeric,engagement_rate,median fill,True
9,numeric,scroll_rate,median fill,True


All selected features available before prediction: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# I will check the feature list for label-derived fields, future-looking fields, product flags, client identifiers, URLs, and private queries. The target is trend_direction, so trend_direction and trend_pct must not appear in the feature matrix. Any future-window or outcome-derived field will be excluded.


In [16]:
target_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

future_or_product_fields = [
    "future_trend",
    "product_flag",
    "refresh_flag",
    "action_taken",
    "url",
    "domain",
    "private_query",
]

feature_columns = list(X.columns)

leakage_columns = [
    col for col in feature_columns
    if any(term in col.lower() for term in [
        "trend",
        "future",
        "flag",
        "action",
        "outcome",
        "label"
    ])
]

target_present_in_features = [
    field for field in target_fields
    if field in feature_columns
]

print("Feature columns checked:", len(feature_columns))
print("Possible leakage columns:", leakage_columns)
print("Target fields inside features:", target_present_in_features)

print(
    "Leakage check passed:",
    len(leakage_columns) == 0 and len(target_present_in_features) == 0
)


Feature columns checked: 27
Possible leakage columns: ['main_intent_transactional']
Target fields inside features: []
Leakage check passed: False


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# I excluded trend_direction and trend_pct because they are outcome-related and could leak the target into the features. I excluded future-window fields because they would not be available at prediction time. I excluded product flags and action fields because they may encode an existing decision rather than independent evidence. I also excluded URLs, domains, private queries, and client identifiers to reduce privacy risk and avoid memorization.


In [18]:
excluded_fields = {
    "trend_direction": "Target/outcome field; using it would directly reveal the label.",
    "trend_pct": "Outcome-derived field; may cause target leakage.",
    "future_trend": "Future information unavailable at prediction time.",
    "product_flag": "Existing product decision, not an independent feature.",
    "refresh_flag": "May encode an existing action or decision.",
    "action_taken": "Post-decision information and unavailable before prediction.",
    "url": "Excluded for privacy and memorization risk.",
    "domain": "Excluded for privacy and memorization risk.",
    "private_query": "Excluded because it may contain sensitive information.",
    "client_id": "Excluded to reduce client memorization and privacy risk.",
}

excluded_summary = pd.DataFrame(
    list(excluded_fields.items()),
    columns=["field", "reason"]
)

display(excluded_summary)

print("Excluded fields documented:", len(excluded_summary))


,field,reason
0,trend_direction,Target/outcome field; using it would directly ...
1,trend_pct,Outcome-derived field; may cause target leakage.
2,future_trend,Future information unavailable at prediction t...
3,product_flag,"Existing product decision, not an independent ..."
4,refresh_flag,May encode an existing action or decision.
5,action_taken,Post-decision information and unavailable befo...
6,url,Excluded for privacy and memorization risk.
7,domain,Excluded for privacy and memorization risk.
8,private_query,Excluded because it may contain sensitive info...
9,client_id,Excluded to reduce client memorization and pri...


Excluded fields documented: 10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.